# GROMACS Lysozyme Tutorial

This notebook follows the GROMACS lysozyme tutorial from http://www.mdtutorials.com/gmx/lysozyme/

We will simulate hen egg white lysozyme (PDB code 1AKI) in water using the CHARMM36 force field.

## Tutorial Steps:
1. Prepare the topology (pdb2gmx)
2. Define the box and add solvent (editconf, solvate)
3. Add ions (genion)
4. Energy minimization (EM)
5. NVT equilibration (constant temperature)
6. NPT equilibration (constant temperature and pressure)
7. Production MD simulation
8. Analysis

**Note:** This tutorial requires GROMACS to be installed and available in your PATH.

In [ ]:
import os
import subprocess
from pathlib import Path

# Create working directory
work_dir = Path("lysozyme_tutorial")
work_dir.mkdir(exist_ok=True)
os.chdir(work_dir)

# Create inputs directory for mdp files
inputs_dir = Path("inputs")
inputs_dir.mkdir(exist_ok=True)

print(f"Working directory: {Path.cwd()}")

## Step 1: Download and Prepare the PDB Structure

Download lysozyme structure (1AKI) from RCSB and remove crystal waters.

In [ ]:
# Download PDB file using curl
!curl -s http://files.rcsb.org/download/1AKI.pdb -o 1aki.pdb

# Remove crystal waters (HOH residues)
!grep -v HOH 1aki.pdb > 1AKI_clean.pdb

print("PDB file downloaded and cleaned")
!wc -l 1aki.pdb 1AKI_clean.pdb

## Step 2: Generate Topology with pdb2gmx

Generate the topology, position restraint file, and processed structure.
We'll use force field option 9 (CHARMM36) and TIP3P water model.

In [ ]:
# Run pdb2gmx - select CHARMM36 force field and TIP3P water
# The exact option number may vary, but CHARMM36 should be one of the first options
# We'll try option 1 first, which is typically CHARMM36
!echo "8\n1" | gmx pdb2gmx -f 1AKI_clean.pdb -o 1AKI_processed.gro -water tip3p

print("\nGenerated files:")
!ls -lh 1AKI_processed.gro topol.top posre.itp

**Note:** If you encounter atomtype errors with TIP3P, this is usually due to force field compatibility. CHARMM36 uses different atomtype names than some other force fields. The solution is to ensure we're using the CHARMM-compatible TIP3P water model.

## Step 3: Define the Box and Add Solvent

Create a cubic box with 1.2 nm from protein to box edge, then fill with water.

In [ ]:
# Define the box
!gmx editconf -f 1AKI_processed.gro -o 1AKI_newbox.gro -c -d 1.2 -bt cubic

print("\nBox defined successfully")

In [ ]:
# Add solvent (water)
!gmx solvate -cp 1AKI_newbox.gro -cs spc216.gro -o 1AKI_solv.gro -p topol.top

print("\nSolvated system created")
print("\nTopology molecules section:")
!grep -A 5 "\[ molecules \]" topol.top

## Step 4: Add Ions for Charge Neutralization

Add ions to neutralize the system charge. First create an MDP file for generating the TPR.

In [ ]:
# Create ions.mdp file
ions_mdp = """
; ions.mdp - used as input into grompp to generate ions.tpr
; Parameters describing what to do, when to stop and what to save
integrator  = steep         ; Algorithm (steep = steepest descent minimization)
emtol       = 1000.0        ; Stop minimization when the maximum force < 1000.0 kJ/mol/nm
emstep      = 0.01          ; Minimization step size
nsteps      = 50000         ; Maximum number of (minimization) steps to perform

; Parameters describing how to find the neighbors of each atom and how to calculate the interactions
nstlist         = 20        ; Frequency to update the neighbor list and long range forces
cutoff-scheme   = Verlet    ; Buffered neighbor searching
coulombtype     = PME       ; Treatment of long range electrostatic interactions
rcoulomb        = 1.0       ; Short-range electrostatic cut-off
rvdw            = 1.0       ; Short-range Van der Waals cut-off
pbc             = xyz       ; Periodic Boundary Conditions in all 3 dimensions
"""

with open("inputs/ions.mdp", "w") as f:
    f.write(ions_mdp)

print("ions.mdp file created")

In [ ]:
# Assemble TPR file
!gmx grompp -f inputs/ions.mdp -c 1AKI_solv.gro -p topol.top -o ions.tpr -maxwarn 1

print("\nTPR file generated")

In [ ]:
# Add ions - replace water molecules with ions to neutralize charge
# Select group 13 (SOL) when prompted
!echo "13" | gmx genion -s ions.tpr -o 1AKI_solv_ions.gro -p topol.top -pname NA -nname CL -neutral

print("\nIons added")
print("\nUpdated topology molecules section:")
!grep -A 5 "\[ molecules \]" topol.top

## Step 5: Energy Minimization

Remove steric clashes and relax the structure before dynamics.

In [ ]:
# Create minim.mdp file
minim_mdp = """
; minim.mdp - used for energy minimization

integrator  = steep         ; Algorithm (steep = steepest descent minimization)
emtol       = 1000.0        ; Stop minimization when the maximum force < 1000.0 kJ/mol/nm
emstep      = 0.01          ; Minimization step size
nsteps      = 50000         ; Maximum number of (minimization) steps to perform

; Parameters describing how to find the neighbors of each atom and how to calculate the interactions
nstlist         = 20        ; Frequency to update the neighbor list
cutoff-scheme   = Verlet    ; Buffered neighbor searching
coulombtype     = PME       ; Treatment of long range electrostatic interactions
rcoulomb        = 1.0       ; Short-range electrostatic cut-off
rvdw            = 1.0       ; Short-range Van der Waals cut-off
pbc             = xyz       ; Periodic Boundary Conditions
"""

with open("inputs/minim.mdp", "w") as f:
    f.write(minim_mdp)

print("minim.mdp file created")

In [ ]:
# Prepare for energy minimization
!gmx grompp -f inputs/minim.mdp -c 1AKI_solv_ions.gro -p topol.top -o em.tpr

print("\nReady for energy minimization")

In [ ]:
# Run energy minimization
!gmx mdrun -v -deffnm em

print("\nEnergy minimization complete")

In [ ]:
# Extract potential energy
!echo "11 0" | gmx energy -f em.edr -o potential.xvg

print("\nPotential energy extracted to potential.xvg")

## Step 6: NVT Equilibration (Temperature Equilibration)

Equilibrate the system at constant volume and temperature (298 K).

In [ ]:
# Create nvt.mdp file
nvt_mdp = """
; nvt.mdp - NVT equilibration

define                  = -DPOSRES  ; position restrain the protein
; Run parameters
integrator              = md        ; leap-frog integrator
nsteps                  = 50000     ; 2 * 50000 = 100 ps
dt                      = 0.002     ; 2 fs
; Output control
nstxout                 = 500       ; save coordinates every 1.0 ps
nstvout                 = 500       ; save velocities every 1.0 ps
nstenergy               = 500       ; save energies every 1.0 ps
nstlog                  = 500       ; update log file every 1.0 ps
; Bond parameters
continuation            = no        ; first dynamics run
constraint_algorithm    = lincs     ; holonomic constraints
constraints             = h-bonds   ; bonds involving H are constrained
lincs_iter              = 1         ; accuracy of LINCS
lincs_order             = 4         ; also related to accuracy
; Nonbonded settings
cutoff-scheme           = Verlet    ; Buffered neighbor searching
nstlist                 = 10        ; 20 fs, largely irrelevant with Verlet
rcoulomb                = 1.0       ; short-range electrostatic cutoff (in nm)
rvdw                    = 1.0       ; short-range van der Waals cutoff (in nm)
DispCorr                = EnerPres  ; account for cut-off vdW scheme
; Electrostatics
coulombtype             = PME       ; Particle Mesh Ewald for long-range electrostatics
pme_order               = 4         ; cubic interpolation
fourierspacing          = 0.16      ; grid spacing for FFT
; Temperature coupling
tcoupl                  = V-rescale             ; modified Berendsen thermostat
tc-grps                 = Protein Non-Protein   ; two coupling groups - more accurate
tau_t                   = 0.1     0.1           ; time constant, in ps
ref_t                   = 300     300           ; reference temperature, one for each group, in K
; Pressure coupling
pcoupl                  = no        ; no pressure coupling in NVT
; Periodic boundary conditions
pbc                     = xyz       ; 3-D PBC
; Velocity generation
gen_vel                 = yes       ; assign velocities from Maxwell distribution
gen_temp                = 300       ; temperature for Maxwell distribution
gen_seed                = -1        ; generate a random seed
"""

with open("inputs/nvt.mdp", "w") as f:
    f.write(nvt_mdp)

print("nvt.mdp file created")

In [ ]:
# Prepare NVT equilibration
!gmx grompp -f inputs/nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr

print("\nReady for NVT equilibration")

In [ ]:
# Run NVT equilibration
!gmx mdrun -deffnm nvt

print("\nNVT equilibration complete")

In [ ]:
# Extract temperature
!echo "16 0" | gmx energy -f nvt.edr -o temperature.xvg

print("\nTemperature data extracted to temperature.xvg")

## Step 7: NPT Equilibration (Pressure Equilibration)

Equilibrate the system at constant temperature and pressure.

In [ ]:
# Create npt.mdp file
npt_mdp = """
; npt.mdp - NPT equilibration

define                  = -DPOSRES  ; position restrain the protein
; Run parameters
integrator              = md        ; leap-frog integrator
nsteps                  = 50000     ; 2 * 50000 = 100 ps
dt                      = 0.002     ; 2 fs
; Output control
nstxout                 = 500       ; save coordinates every 1.0 ps
nstvout                 = 500       ; save velocities every 1.0 ps
nstenergy               = 500       ; save energies every 1.0 ps
nstlog                  = 500       ; update log file every 1.0 ps
; Bond parameters
continuation            = yes       ; continuing from NVT
constraint_algorithm    = lincs     ; holonomic constraints
constraints             = h-bonds   ; bonds involving H are constrained
lincs_iter              = 1         ; accuracy of LINCS
lincs_order             = 4         ; also related to accuracy
; Nonbonded settings
cutoff-scheme           = Verlet    ; Buffered neighbor searching
nstlist                 = 10        ; 20 fs, largely irrelevant with Verlet scheme
rcoulomb                = 1.0       ; short-range electrostatic cutoff (in nm)
rvdw                    = 1.0       ; short-range van der Waals cutoff (in nm)
DispCorr                = EnerPres  ; account for cut-off vdW scheme
; Electrostatics
coulombtype             = PME       ; Particle Mesh Ewald for long-range electrostatics
pme_order               = 4         ; cubic interpolation
fourierspacing          = 0.16      ; grid spacing for FFT
; Temperature coupling
tcoupl                  = V-rescale             ; modified Berendsen thermostat
tc-grps                 = Protein Non-Protein   ; two coupling groups
tau_t                   = 0.1     0.1           ; time constant, in ps
ref_t                   = 300     300           ; reference temperature, one for each group, in K
; Pressure coupling
pcoupl                  = Parrinello-Rahman     ; Pressure coupling on in NPT
pcoupltype              = isotropic             ; uniform scaling of box vectors
tau_p                   = 2.0                   ; time constant, in ps
ref_p                   = 1.0                   ; reference pressure, in bar
compressibility         = 4.5e-5                ; isothermal compressibility of water, bar^-1
refcoord_scaling        = com
; Periodic boundary conditions
pbc                     = xyz       ; 3-D PBC
; Velocity generation
gen_vel                 = no        ; velocity generated during NVT
"""

with open("inputs/npt.mdp", "w") as f:
    f.write(npt_mdp)

print("npt.mdp file created")

In [ ]:
# Prepare NPT equilibration
!gmx grompp -f inputs/npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr

print("\nReady for NPT equilibration")

In [ ]:
# Run NPT equilibration
!gmx mdrun -deffnm npt

print("\nNPT equilibration complete")

In [ ]:
# Extract pressure
!echo "17 0" | gmx energy -f npt.edr -o pressure.xvg

print("\nPressure data extracted to pressure.xvg")

In [ ]:
# Extract density
!echo "23 0" | gmx energy -f npt.edr -o density.xvg

print("\nDensity data extracted to density.xvg")

## Step 8: Production MD Simulation

Run a 1 ns production MD simulation without position restraints.

In [ ]:
# Create md.mdp file for production MD
md_mdp = """
; md.mdp - production MD simulation

; Run parameters
integrator              = md        ; leap-frog integrator
nsteps                  = 500000    ; 2 * 500000 = 1000 ps (1 ns)
dt                      = 0.002     ; 2 fs
; Output control
nstxout                 = 0         ; suppress bulky .trr file by specifying
nstvout                 = 0         ; 0 for output frequency of nstxout,
nstfout                 = 0         ; nstvout, and nstfout
nstenergy               = 5000      ; save energies every 10.0 ps
nstlog                  = 5000      ; update log file every 10.0 ps
nstxout-compressed      = 5000      ; save compressed coordinates every 10.0 ps
compressed-x-grps       = System    ; save the whole system
; Bond parameters
continuation            = yes       ; continuing from NPT
constraint_algorithm    = lincs     ; holonomic constraints
constraints             = h-bonds   ; bonds involving H are constrained
lincs_iter              = 1         ; accuracy of LINCS
lincs_order             = 4         ; also related to accuracy
; Nonbonded settings
cutoff-scheme           = Verlet    ; Buffered neighbor searching
nstlist                 = 10        ; 20 fs, largely irrelevant with Verlet
rcoulomb                = 1.0       ; short-range electrostatic cutoff (in nm)
rvdw                    = 1.0       ; short-range van der Waals cutoff (in nm)
DispCorr                = EnerPres  ; account for cut-off vdW scheme
; Electrostatics
coulombtype             = PME       ; Particle Mesh Ewald for long-range electrostatics
pme_order               = 4         ; cubic interpolation
fourierspacing          = 0.16      ; grid spacing for FFT
; Temperature coupling
tcoupl                  = V-rescale             ; modified Berendsen thermostat
tc-grps                 = Protein Non-Protein   ; two coupling groups
tau_t                   = 0.1     0.1           ; time constant, in ps
ref_t                   = 300     300           ; reference temperature, one for each group, in K
; Pressure coupling
pcoupl                  = Parrinello-Rahman     ; Pressure coupling on
pcoupltype              = isotropic             ; uniform scaling of box vectors
tau_p                   = 2.0                   ; time constant, in ps
ref_p                   = 1.0                   ; reference pressure, in bar
compressibility         = 4.5e-5                ; isothermal compressibility of water, bar^-1
; Periodic boundary conditions
pbc                     = xyz       ; 3-D PBC
; Dispersion correction
DispCorr                = EnerPres  ; account for cut-off vdW scheme
; Velocity generation
gen_vel                 = no        ; continuing from NPT equilibration
"""

with open("inputs/md.mdp", "w") as f:
    f.write(md_mdp)

print("md.mdp file created")

In [ ]:
# Prepare production MD
!gmx grompp -f inputs/md.mdp -c npt.gro -t npt.cpt -p topol.top -o md_0_10.tpr

print("\nReady for production MD")

In [ ]:
# Run production MD (this will take a while!)
# For GPU acceleration, GROMACS will automatically detect and use available GPUs
!gmx mdrun -deffnm md_0_10

print("\nProduction MD complete!")

## Step 9: Analysis

Analyze the simulation results.

### 9.1: Correct for Periodicity

Process the trajectory to account for periodic boundary conditions.

In [ ]:
# Correct for periodicity - center protein and reimage molecules
# Select 1 (Protein) for centering and 0 (System) for output
!echo "1 0" | gmx trjconv -s md_0_10.tpr -f md_0_10.xtc -o md_0_10_noPBC.xtc -pbc mol -center

print("\nTrajectory corrected for periodicity")

### 9.2: Calculate RMSD

Calculate root-mean-square deviation to measure structural stability.

In [ ]:
# Calculate RMSD relative to the starting structure
# Select 4 (Backbone) for both least-squares fit and RMSD calculation
!echo "4 4" | gmx rms -s md_0_10.tpr -f md_0_10_noPBC.xtc -o rmsd.xvg -tu ns

print("\nRMSD calculated (relative to starting structure)")

In [ ]:
# Calculate RMSD relative to the crystal structure
!echo "4 4" | gmx rms -s em.tpr -f md_0_10_noPBC.xtc -o rmsd_xtal.xvg -tu ns

print("\nRMSD calculated (relative to crystal structure)")

### 9.3: Plot Results

Visualize the analysis results using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def read_xvg(filename):
    """Read GROMACS xvg file and return data arrays."""
    data = []
    with open(filename, 'r') as f:
        for line in f:
            if not line.startswith('#') and not line.startswith('@'):
                data.append([float(x) for x in line.split()])
    return np.array(data)

# Create a figure with multiple subplots
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle('GROMACS Lysozyme Tutorial - Analysis Results', fontsize=16)

# Plot potential energy from EM
try:
    pot_data = read_xvg('potential.xvg')
    axes[0, 0].plot(pot_data[:, 0], pot_data[:, 1])
    axes[0, 0].set_xlabel('Energy Minimization Step')
    axes[0, 0].set_ylabel('Potential Energy (kJ/mol)')
    axes[0, 0].set_title('Energy Minimization')
    axes[0, 0].grid(True, alpha=0.3)
except FileNotFoundError:
    axes[0, 0].text(0.5, 0.5, 'Data not available', ha='center', va='center')

# Plot temperature from NVT
try:
    temp_data = read_xvg('temperature.xvg')
    axes[0, 1].plot(temp_data[:, 0], temp_data[:, 1])
    axes[0, 1].axhline(y=298, color='r', linestyle='--', label='Target (298 K)')
    axes[0, 1].set_xlabel('Time (ps)')
    axes[0, 1].set_ylabel('Temperature (K)')
    axes[0, 1].set_title('NVT Equilibration - Temperature')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
except FileNotFoundError:
    axes[0, 1].text(0.5, 0.5, 'Data not available', ha='center', va='center')

# Plot pressure from NPT
try:
    press_data = read_xvg('pressure.xvg')
    axes[1, 0].plot(press_data[:, 0], press_data[:, 1], alpha=0.5)
    # Calculate running average
    window = 50
    running_avg = np.convolve(press_data[:, 1], np.ones(window)/window, mode='valid')
    axes[1, 0].plot(press_data[window-1:, 0], running_avg, 'r-', linewidth=2, label='Running Average')
    axes[1, 0].axhline(y=1, color='g', linestyle='--', label='Target (1 bar)')
    axes[1, 0].set_xlabel('Time (ps)')
    axes[1, 0].set_ylabel('Pressure (bar)')
    axes[1, 0].set_title('NPT Equilibration - Pressure')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
except FileNotFoundError:
    axes[1, 0].text(0.5, 0.5, 'Data not available', ha='center', va='center')

# Plot density from NPT
try:
    dens_data = read_xvg('density.xvg')
    axes[1, 1].plot(dens_data[:, 0], dens_data[:, 1], alpha=0.5)
    # Calculate running average
    running_avg = np.convolve(dens_data[:, 1], np.ones(window)/window, mode='valid')
    axes[1, 1].plot(dens_data[window-1:, 0], running_avg, 'r-', linewidth=2, label='Running Average')
    axes[1, 1].set_xlabel('Time (ps)')
    axes[1, 1].set_ylabel('Density (kg/m³)')
    axes[1, 1].set_title('NPT Equilibration - Density')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
except FileNotFoundError:
    axes[1, 1].text(0.5, 0.5, 'Data not available', ha='center', va='center')

# Plot RMSD (starting structure)
try:
    rmsd_data = read_xvg('rmsd.xvg')
    axes[2, 0].plot(rmsd_data[:, 0], rmsd_data[:, 1])
    axes[2, 0].set_xlabel('Time (ns)')
    axes[2, 0].set_ylabel('RMSD (nm)')
    axes[2, 0].set_title('RMSD (vs. Starting Structure)')
    axes[2, 0].grid(True, alpha=0.3)
except FileNotFoundError:
    axes[2, 0].text(0.5, 0.5, 'Data not available', ha='center', va='center')

# Plot RMSD (crystal structure)
try:
    rmsd_xtal_data = read_xvg('rmsd_xtal.xvg')
    axes[2, 1].plot(rmsd_xtal_data[:, 0], rmsd_xtal_data[:, 1])
    axes[2, 1].set_xlabel('Time (ns)')
    axes[2, 1].set_ylabel('RMSD (nm)')
    axes[2, 1].set_title('RMSD (vs. Crystal Structure)')
    axes[2, 1].grid(True, alpha=0.3)
except FileNotFoundError:
    axes[2, 1].text(0.5, 0.5, 'Data not available', ha='center', va='center')

plt.tight_layout()
plt.savefig('analysis_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nAnalysis plots saved to analysis_summary.png")

### 10.1: Radius of Gyration (Compactness Analysis)

The radius of gyration is a measure of protein compactness. A stably folded protein maintains a relatively steady Rg value, while unfolding causes Rg to increase over time.

In [ ]:
# Calculate radius of gyration for the protein
!echo "1" | gmx gyrate -s md_0_10.tpr -f md_0_10_noPBC.xtc -o gyrate.xvg -tu ns

print("\nRadius of gyration calculated and saved to gyrate.xvg")

### 10.2: Secondary Structure Analysis

Analyze the persistence of secondary structure elements (α-helices, β-sheets, etc.) using the DSSP algorithm.

In [ ]:
# Analyze secondary structure using DSSP
!gmx dssp -s md_0_10.tpr -f md_0_10_noPBC.xtc -tu ns -o dssp.dat -num dssp_num.xvg

print("\nSecondary structure analysis complete:")
print("- dssp.dat: Secondary structure assignments per residue per frame")
print("- dssp_num.xvg: Number of residues in each secondary structure type vs time")

### 10.3: Hydrogen Bond Analysis

Analyze hydrogen bonds within the protein and between protein and water. GROMACS uses donor-H distance ≤ 0.35 nm and donor-acceptor-H angle ≤ 30° criteria.

In [ ]:
# Hydrogen bonds within protein backbone (MainChain+H includes N, Cα, C, O, and amide H)
!echo "7 7" | gmx hbond -s md_0_10.tpr -f md_0_10_noPBC.xtc -tu ns -num hbnum_mainchain.xvg

print("\nBackbone hydrogen bonds analysis complete")

In [ ]:
# Hydrogen bonds among sidechain atoms
!echo "8 8" | gmx hbond -s md_0_10.tpr -f md_0_10_noPBC.xtc -tu ns -num hbnum_sidechain.xvg

print("\nSidechain hydrogen bonds analysis complete")

In [ ]:
# Hydrogen bonds between protein and water
!echo "1 13" | gmx hbond -s md_0_10.tpr -f md_0_10_noPBC.xtc -tu ns -num hbnum_prot_wat.xvg

print("\nProtein-water hydrogen bonds analysis complete")

### 10.4: Root Mean Square Fluctuation (RMSF)

Calculate per-residue flexibility by analyzing the fluctuation of each residue around its average position.

In [ ]:
# Calculate RMSF for protein residues
!echo "1" | gmx rmsf -s md_0_10.tpr -f md_0_10_noPBC.xtc -o rmsf.xvg -res

print("\nRMSF analysis complete - shows per-residue flexibility")
print("Higher RMSF values indicate more flexible regions")

### 10.5: Advanced Analysis Plots

Visualize the additional analysis results.

In [ ]:
# Plot advanced analysis results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('GROMACS Lysozyme Tutorial - Advanced Analysis Results', fontsize=16)

# Plot radius of gyration
try:
    rg_data = read_xvg('gyrate.xvg')
    axes[0, 0].plot(rg_data[:, 0], rg_data[:, 1])
    axes[0, 0].set_xlabel('Time (ns)')
    axes[0, 0].set_ylabel('Rg (nm)')
    axes[0, 0].set_title('Radius of Gyration')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Calculate and display average Rg
    avg_rg = np.mean(rg_data[:, 1])
    std_rg = np.std(rg_data[:, 1])
    axes[0, 0].axhline(y=avg_rg, color='r', linestyle='--', 
                       label=f'Average: {avg_rg:.3f} ± {std_rg:.3f} nm')
    axes[0, 0].legend()
except FileNotFoundError:
    axes[0, 0].text(0.5, 0.5, 'Rg data not available', ha='center', va='center')

# Plot RMSF
try:
    rmsf_data = read_xvg('rmsf.xvg')
    axes[0, 1].plot(rmsf_data[:, 0], rmsf_data[:, 1])
    axes[0, 1].set_xlabel('Residue Number')
    axes[0, 1].set_ylabel('RMSF (nm)')
    axes[0, 1].set_title('Root Mean Square Fluctuation')
    axes[0, 1].grid(True, alpha=0.3)
except FileNotFoundError:
    axes[0, 1].text(0.5, 0.5, 'RMSF data not available', ha='center', va='center')

# Plot hydrogen bonds - backbone
try:
    hb_main_data = read_xvg('hbnum_mainchain.xvg')
    axes[1, 0].plot(hb_main_data[:, 0], hb_main_data[:, 1], label='Backbone', alpha=0.7)
    
    # Try to add sidechain data
    try:
        hb_side_data = read_xvg('hbnum_sidechain.xvg')
        axes[1, 0].plot(hb_side_data[:, 0], hb_side_data[:, 1], label='Sidechain', alpha=0.7)
    except FileNotFoundError:
        pass
        
    axes[1, 0].set_xlabel('Time (ns)')
    axes[1, 0].set_ylabel('Number of H-bonds')
    axes[1, 0].set_title('Intramolecular Hydrogen Bonds')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
except FileNotFoundError:
    axes[1, 0].text(0.5, 0.5, 'H-bond data not available', ha='center', va='center')

# Plot protein-water hydrogen bonds
try:
    hb_pw_data = read_xvg('hbnum_prot_wat.xvg')
    axes[1, 1].plot(hb_pw_data[:, 0], hb_pw_data[:, 1], color='blue', alpha=0.7)
    axes[1, 1].set_xlabel('Time (ns)')
    axes[1, 1].set_ylabel('Number of H-bonds')
    axes[1, 1].set_title('Protein-Water Hydrogen Bonds')
    axes[1, 1].grid(True, alpha=0.3)
    
    # Calculate average
    avg_hb = np.mean(hb_pw_data[:, 1])
    axes[1, 1].axhline(y=avg_hb, color='r', linestyle='--', 
                       label=f'Average: {avg_hb:.1f}')
    axes[1, 1].legend()
except FileNotFoundError:
    axes[1, 1].text(0.5, 0.5, 'Protein-water H-bond\ndata not available', ha='center', va='center')

plt.tight_layout()
plt.savefig('advanced_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nAdvanced analysis plots saved to advanced_analysis.png")

## Updated Summary

This notebook now includes the complete GROMACS lysozyme tutorial with advanced analysis:

### Basic Setup and Simulation:
1. ✅ Downloaded and cleaned the PDB structure (1AKI)
2. ✅ Generated topology with pdb2gmx using CHARMM36 force field
3. ✅ Defined the simulation box and solvated with water
4. ✅ Added ions to neutralize the system
5. ✅ Performed energy minimization
6. ✅ NVT equilibration (temperature stabilization)
7. ✅ NPT equilibration (pressure/density stabilization)
8. ✅ Production MD simulation (1 ns)

### Basic Analysis:
9. ✅ Trajectory correction for periodicity
10. ✅ RMSD analysis (vs starting and crystal structures)
11. ✅ Basic plots (energy, temperature, pressure, density)

### Advanced Analysis (New):
12. ✅ Radius of gyration (protein compactness)
13. ✅ Secondary structure analysis (DSSP)
14. ✅ Hydrogen bond analysis:
    - Backbone hydrogen bonds
    - Sidechain hydrogen bonds  
    - Protein-water hydrogen bonds
15. ✅ Root mean square fluctuation (per-residue flexibility)

### Key Output Files

**Basic simulation files:**
- `topol.top` - System topology
- `md_0_10.xtc` - Production trajectory (compressed)
- `md_0_10_noPBC.xtc` - Corrected trajectory for analysis

**Basic analysis files:**
- `rmsd.xvg`, `rmsd_xtal.xvg` - RMSD data
- `temperature.xvg`, `pressure.xvg`, `density.xvg` - Equilibration data
- `analysis_summary.png` - Basic analysis plots

**Advanced analysis files:**
- `gyrate.xvg` - Radius of gyration vs time
- `dssp.dat` - Secondary structure assignments
- `dssp_num.xvg` - Secondary structure counts vs time
- `hbnum_mainchain.xvg` - Backbone hydrogen bonds
- `hbnum_sidechain.xvg` - Sidechain hydrogen bonds  
- `hbnum_prot_wat.xvg` - Protein-water hydrogen bonds
- `rmsf.xvg` - Per-residue flexibility
- `advanced_analysis.png` - Advanced analysis plots

### Interpretation Guidelines

- **RMSD**: Should stabilize around 0.1-0.3 nm for a stable protein
- **Radius of gyration**: Should remain relatively constant (~1.4 nm for lysozyme)
- **Temperature/Pressure**: Should converge to target values during equilibration
- **Hydrogen bonds**: Backbone ~55, sidechain ~20, protein-water ~55 for lysozyme
- **RMSF**: Loop regions typically show higher flexibility than secondary structure elements
